# CityFlo Bus Service Metro Cities
## Data Understanding, Data Cleaning and Exploratory Data Analysis using Python

**Tools Used:** Python, Pandas, NumPy, Matplotlib, Seaborn

---

# 1. Introduction

Public transportation systems generate large volumes of operational and customer-related data every day. Analyzing this information helps transport operators understand booking behavior, passenger preferences, trip performance, revenue generation, and service quality.

The CityFlo Bus Service Metro Cities dataset contains detailed information about bus bookings, customer demographics, travel characteristics, payment methods, booking channels, operational performance, and customer feedback. These records provide valuable insights into both customer behavior and business operations.

The primary objective of this project is to perform comprehensive data understanding, data cleaning, and exploratory data analysis (EDA) to transform the raw dataset into a high-quality analytical dataset. The preprocessing pipeline is designed to be reusable and robust, enabling it to handle similar datasets with minimal modification.

This notebook follows a systematic workflow consisting of:

- Data Understanding
- Data Cleaning
- Exploratory Data Analysis
- Business Insights

The cleaned dataset obtained through this workflow can be further utilized for visualization, business intelligence, statistical analysis, or predictive machine learning applications.

# 2. Objectives

The objectives of this project are:

- Understand the overall structure and characteristics of the dataset.
- Identify data quality issues such as missing values, duplicates, inconsistent formats, and incorrect data types.
- Build a reusable preprocessing pipeline capable of handling similar transportation datasets.
- Standardize and clean the dataset to improve reliability.
- Explore customer booking behavior and operational trends.
- Generate meaningful business insights through exploratory data analysis.
- Prepare a clean dataset suitable for further predictive analytics and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 150)
print("Libraries imported successfully.")

# 3. Dataset Loading

The dataset is imported into a Pandas DataFrame for analysis. Once loaded, the first few records are displayed to verify that the dataset has been imported correctly and to obtain an initial understanding of its structure.

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/rishijmanna/cityflo-bus-service-metro-cities-cleaned/cityflo_bus_service_metro_cities_cleaned.csv")
df.head()

# 4. Dataset Overview

Understanding the overall characteristics of a dataset is the first step before performing preprocessing or analysis. This section summarizes the dataset size, structure, memory usage, and overall completeness.

In [ ]:
print("CITYFLO DATASET OVERVIEW")

print(f"Number of Observations : {df.shape[0]:,}")
print(f"Number of Features     : {df.shape[1]}")

print(f"\nMemory Usage : {df.memory_usage(deep=True).sum()/(1024**2):.2f} MB")

print(f"\nDuplicate Rows : {df.duplicated().sum()}")

print(f"Missing Values : {df.isnull().sum().sum()}")

print(f"Total Cells : {df.size:,}")

print(f"Missing Percentage : {(df.isnull().sum().sum()/df.size)*100:.2f}%")

# Dataset Snapshot

The following sample records provide an overview of the information contained in the dataset. Reviewing representative observations helps verify successful data loading and offers an initial understanding of the available features.

In [ ]:
display(df.head(10))

# Random Sample of Records

Displaying randomly selected observations provides a more representative view of the dataset and helps identify potential inconsistencies that may not appear in the first few records.

In [ ]:
display(df.sample(10, random_state=42))

# Dataset Dimensions

The dimensions of a dataset indicate the number of observations (rows) and features (columns). Understanding the dataset size helps estimate computational requirements and provides context for subsequent preprocessing and analysis.

In [ ]:
rows, columns = df.shape

dimension_summary = pd.DataFrame({
    "Property": [
        "Number of Rows",
        "Number of Columns"
    ],
    "Value": [
        rows,
        columns
    ]
})

display(dimension_summary)

# 5. Data Understanding

## 5.1 Dataset Information

Before performing any preprocessing, it is important to understand the structure of the dataset. This section provides detailed information about each feature, including its data type, number of non-null observations, and memory usage.

Understanding the dataset structure helps identify potential preprocessing requirements such as missing values, incorrect data types, and memory optimization opportunities.

In [ ]:
print("DATASET INFORMATION")
df.info()

## 5.2 Memory Usage Analysis

Memory optimization is an important aspect of data preprocessing, particularly when working with large datasets. This section reports the memory consumed by each feature as well as the total memory occupied by the dataset.

Understanding memory utilization helps identify opportunities for data type optimization during the cleaning stage.

In [ ]:
memory_df = pd.DataFrame({

    "Column": df.columns,

    "Memory (KB)": (
        df.memory_usage(deep=True)[1:] / 1024
    ).round(2)

})

display(memory_df.sort_values(
    "Memory (KB)",
    ascending=False
))

## 5.3 Feature Metadata

A metadata summary provides a comprehensive overview of every feature within the dataset. It includes information about data types, missing values, unique observations, and representative sample values.

This information serves as the foundation for identifying preprocessing requirements and understanding the nature of each feature.

In [ ]:
metadata = pd.DataFrame({

    "Feature": df.columns,

    "Data Type": df.dtypes.astype(str).values,

    "Non-Null": df.count().values,

    "Missing": df.isnull().sum().values,

    "Missing (%)": (
        df.isnull().mean()*100
    ).round(2).values,

    "Unique Values": df.nunique().values,

    "Sample Values":[

        ", ".join(
            map(str,
                df[col].dropna().unique()[:3]
            )
        )

        for col in df.columns

    ]

})

display(metadata)

## 5.4 Automatic Feature Classification

Each feature serves a different analytical purpose. Automatically classifying features improves preprocessing decisions and helps determine appropriate analytical techniques.

The features are categorized into:

- Identifier Features
- Numerical Features
- Categorical Features
- Date/Time Features
- Boolean Features

In [ ]:
feature_type = []

for col in df.columns:

    if df[col].nunique() == len(df):

        category = "Identifier"

    elif pd.api.types.is_numeric_dtype(df[col]):

        category = "Numerical"

    elif pd.api.types.is_datetime64_any_dtype(df[col]):

        category = "Datetime"

    elif df[col].dropna().isin([True,False]).all():

        category = "Boolean"

    else:

        category = "Categorical"

    feature_type.append(category)

feature_summary = pd.DataFrame({

    "Feature": df.columns,

    "Category": feature_type,

    "Data Type": df.dtypes.astype(str)

})

display(feature_summary)

## 5.5 Feature Distribution Summary

This section summarizes the distribution of feature types within the dataset. Understanding the composition of the dataset provides an overview of the variables available for analysis and guides subsequent preprocessing decisions.

In [ ]:
feature_summary["Category"].value_counts().to_frame(
    "Number of Features"
)

## 5.6 Statistical Summary of Numerical Features

Descriptive statistics provide a concise summary of the central tendency, variability, and overall distribution of numerical variables.

The following statistics are reported:

- Count
- Mean
- Standard Deviation
- Minimum
- Quartiles
- Maximum

These measures assist in detecting anomalies, extreme values, and potential preprocessing requirements.

In [ ]:
display(df.describe().T)

## 5.7 Statistical Summary of Categorical Features

Categorical variables are summarized to identify their frequency distributions, dominant categories, and diversity.

Understanding categorical features assists in detecting inconsistent labels, uncommon categories, and potential standardization requirements.

In [ ]:
display(
    df.describe(include="object").T
)

## 5.8 Unique Value Summary

The number of distinct values contained within each feature provides insight into its characteristics. Features containing a single unique value may contribute little analytical information, whereas features with very large numbers of unique values often represent identifiers or free-text attributes.

This summary helps identify features requiring further preprocessing.

In [ ]:
unique_summary = pd.DataFrame({

    "Feature": df.columns,

    "Unique Values":[

        df[col].nunique()

        for col in df.columns

    ]

})

display(
    unique_summary.sort_values(
        "Unique Values",
        ascending=False
    )
)

## 5.9 Initial Data Quality Report

Before applying any preprocessing techniques, an initial quality assessment is performed to evaluate the completeness and integrity of the dataset.

The report summarizes:

- Dataset dimensions
- Missing values
- Duplicate observations
- Feature composition
- Memory usage

This baseline assessment provides a reference for measuring the effectiveness of subsequent data cleaning operations.

In [ ]:
quality_report = pd.DataFrame({

    "Metric":[

        "Rows",

        "Columns",

        "Duplicate Rows",

        "Missing Cells",

        "Memory Usage (MB)",

        "Numerical Features",

        "Categorical Features"

    ],

    "Value":[

        len(df),

        len(df.columns),

        df.duplicated().sum(),

        df.isnull().sum().sum(),

        round(
            df.memory_usage(deep=True).sum()/(1024**2),
            2
        ),

        len(
            df.select_dtypes(
                include=np.number
            ).columns
        ),

        len(
            df.select_dtypes(
                include="object"
            ).columns
        )

    ]

})

display(quality_report)

# 6. Data Quality Assessment

## 6.1 Missing Value Analysis

Missing values are one of the most common data quality issues in real-world datasets. They may occur due to incomplete data collection, system failures, manual entry errors, or unavailable information.

This section examines the extent of missing data in each feature by reporting both the number and percentage of missing values. Understanding missing data patterns helps determine appropriate preprocessing strategies during the data cleaning phase.

In [ ]:
missing_summary = pd.DataFrame({
    "Feature": df.columns,
    "Missing Values": df.isnull().sum().values,
    "Missing (%)": (
        df.isnull().mean() * 100
    ).round(2).values
})

missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

display(missing_summary)

In [ ]:
missing_only = missing_summary[
    missing_summary["Missing Values"] > 0
]

if missing_only.empty:
    print("✅ No missing values detected.")
else:
    display(missing_only)

## 6.2 Duplicate Record Analysis

Duplicate records may arise due to repeated data collection, synchronization issues, or manual entry errors.

Duplicate observations can distort statistical summaries, introduce bias into analytical models, and increase storage requirements. Therefore, identifying duplicate records is an essential step before performing any preprocessing operations.

In [ ]:
duplicate_rows = df.duplicated().sum()

duplicate_percentage = (
    duplicate_rows / len(df)
) * 100

duplicate_report = pd.DataFrame({

    "Metric": [

        "Duplicate Rows",

        "Duplicate Percentage (%)"

    ],

    "Value": [

        duplicate_rows,

        round(duplicate_percentage,2)

    ]

})

display(duplicate_report)

In [ ]:
if duplicate_rows > 0:

    display(df[df.duplicated()].head())

else:

    print("✅ No duplicate rows found.")

## 6.3 Duplicate Feature Detection

Occasionally, datasets may contain multiple columns storing identical information under different names. Such duplicate features increase memory usage and may negatively affect analysis.

This section checks whether any pair of features contains exactly the same values throughout the dataset.

In [ ]:
duplicate_columns = []

columns = df.columns

for i in range(len(columns)):

    for j in range(i+1, len(columns)):

        if df[columns[i]].equals(df[columns[j]]):

            duplicate_columns.append(
                (columns[i], columns[j])
            )

if duplicate_columns:

    display(pd.DataFrame(
        duplicate_columns,
        columns=[
            "Feature 1",
            "Feature 2"
        ]
    ))

else:

    print("✅ No duplicate columns detected.")

## 6.4 Constant Feature Detection

Constant features contain only a single unique value throughout the dataset. Since these variables exhibit no variation, they provide little or no analytical value and are typically removed during preprocessing.

In [ ]:
constant_columns = []

for col in df.columns:

    if df[col].nunique(dropna=False) == 1:

        constant_columns.append(col)

constant_df = pd.DataFrame({

    "Constant Features": constant_columns

})

display(constant_df)

## 6.5 Quasi-Constant Feature Detection

A quasi-constant feature contains one dominant value appearing in the vast majority of observations. Although not completely constant, such variables often contribute very little useful information.

This section identifies features where a single value accounts for more than 95% of all observations.

In [ ]:
threshold = 0.95

quasi_constant = []

for col in df.columns:

    dominant_ratio = (
        df[col]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .max()
    )

    if dominant_ratio >= threshold:

        quasi_constant.append({

            "Feature": col,

            "Dominant Ratio": round(
                dominant_ratio,
                4
            )

        })

display(pd.DataFrame(quasi_constant))

## 6.6 High Cardinality Detection

Categorical variables containing a very large number of unique categories are referred to as high-cardinality features.

Such variables often represent identifiers, names, or free-text fields. Detecting these features early assists in selecting suitable encoding techniques during later stages of analysis.

high_cardinality = []

for col in df.select_dtypes(include="object"):

    unique = df[col].nunique()

    if unique > 50:

        high_cardinality.append({

            "Feature": col,

            "Unique Values": unique,

            "Unique Ratio (%)": round(
                unique/len(df)*100,
                2
            )

        })

display(
    pd.DataFrame(high_cardinality)
)

## 6.7 Identifier Feature Detection

Identifier features uniquely distinguish individual records rather than describing their characteristics. Examples include booking IDs, customer IDs, transaction IDs, and ticket numbers.

Such features are generally retained for reference but excluded from statistical analysis and predictive modeling.

In [ ]:
identifier_columns = []

for col in df.columns:

    if df[col].nunique() == len(df):

        identifier_columns.append(col)

identifier_df = pd.DataFrame({

    "Identifier Features": identifier_columns

})

display(identifier_df)

## 6.8 Leading and Trailing Whitespace Detection

Leading and trailing spaces are common data entry issues that can cause identical values to be treated as different categories.

This section identifies text features containing unnecessary whitespace so they can be standardized during the cleaning phase.

In [ ]:
whitespace_columns = []

for col in df.select_dtypes(include="object").columns:

    if df[col].dropna().astype(str).str.strip().ne(
        df[col].dropna().astype(str)
    ).any():

        whitespace_columns.append(col)

if whitespace_columns:

    display(pd.DataFrame({
        "Features with Extra Spaces": whitespace_columns
    }))

else:

    print("✅ No leading or trailing whitespace detected.")

## 6.9 Empty String Detection

Empty strings ("") represent missing information that is not recognized as a standard missing value (`NaN`).

This section identifies features containing empty strings so they can be converted into proper missing values during preprocessing.

In [ ]:
empty_summary = []

for col in df.select_dtypes(include="object").columns:

    count = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    if count > 0:

        empty_summary.append({

            "Feature": col,

            "Empty Strings": count

        })


## 6.10 Placeholder Value Detection

Datasets often use placeholder values such as "NA", "N/A", "Unknown", "-", or "Missing" instead of standard null values.

Identifying these placeholders ensures they can be converted into proper missing values during data cleaning.

In [ ]:
placeholders = [

    "NA",
    "N/A",
    "NULL",
    "Null",
    "null",
    "None",
    "none",
    "Unknown",
    "unknown",
    "-",
    "?",
    "Missing",
    "missing"

]

placeholder_summary = []

for col in df.select_dtypes(include="object").columns:

    count = df[col].isin(placeholders).sum()

    if count > 0:

        placeholder_summary.append({

            "Feature": col,

            "Placeholder Count": count

        })

display(pd.DataFrame(placeholder_summary))

## 6.11 Mixed Data Type Detection

A feature should ideally contain values of a single data type. Mixed data types may indicate inconsistent data entry or import issues.

This section identifies columns containing more than one Python data type.

In [ ]:
mixed_columns = []

for col in df.columns:

    types = df[col].dropna().map(type).unique()

    if len(types) > 1:

        mixed_columns.append({

            "Feature": col,

            "Data Types": ", ".join(
                [t.__name__ for t in types]
            )

        })

display(pd.DataFrame(mixed_columns))

## 6.12 Case Inconsistency Detection

Inconsistent capitalization (for example, "Male", "male", and "MALE") creates duplicate categories and affects summary statistics.

This section identifies categorical features that may require case standardization.

In [ ]:
case_columns = []

for col in df.select_dtypes(include="object").columns:

    unique_original = df[col].dropna().nunique()

    unique_lower = (
        df[col]
        .dropna()
        .astype(str)
        .str.lower()
        .nunique()
    )

    if unique_original != unique_lower:

        case_columns.append({

            "Feature": col,

            "Original Unique": unique_original,

            "After Lowercase": unique_lower

        })

display(pd.DataFrame(case_columns))

## 6.13 Hidden Character Detection

Hidden characters such as tabs (`\\t`), newline characters (`\\n`), and carriage returns (`\\r`) may affect text processing and create inconsistent values.

This section checks for the presence of these non-visible characters.

In [ ]:
hidden_summary = []

pattern = r'[\t\n\r]'

for col in df.select_dtypes(include="object").columns:

    count = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(pattern, regex=True)
        .sum()
    )

    if count > 0:

        hidden_summary.append({

            "Feature": col,

            "Affected Records": count

        })

display(pd.DataFrame(hidden_summary))

##  6.14 Numeric Values Stored as Text

Numerical values are sometimes imported as text due to formatting inconsistencies. Such features cannot be used directly for numerical analysis until converted into appropriate numeric data types.

This section identifies object-type features that can potentially be converted into numeric variables.

In [ ]:

numeric_text = []

for col in df.select_dtypes(include="object").columns:

    converted = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    ratio = converted.notna().mean()

    if ratio >= 0.90:

        numeric_text.append({

            "Feature": col,

            "Convertible (%)": round(
                ratio * 100,
                2
            )

        })

display(pd.DataFrame(numeric_text))

## 6.15 Date Format Detection

Date and time variables are essential for temporal analysis. However, dates are often imported as text, preventing direct date-based operations.

This section identifies object-type features that can potentially be converted into datetime format.

In [ ]:
date_columns = []

for col in df.select_dtypes(include="object").columns:

    converted = pd.to_datetime(
        df[col],
        errors="coerce"
    )

    ratio = converted.notna().mean()

    if ratio >= 0.90:

        date_columns.append({

            "Feature": col,

            "Valid Dates (%)": round(
                ratio * 100,
                2
            )

        })

display(pd.DataFrame(date_columns))

# 7. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is performed to understand the characteristics, distributions, relationships, and trends within the cleaned CityFlo Bus Service dataset. Through statistical summaries and visualizations, this analysis identifies customer behavior, booking patterns, operational performance, and revenue-generating factors that support data-driven business decisions.

The EDA is divided into three sections:

- Univariate Analysis
- Bivariate Analysis
- Multivariate Analysis

## 7.1 Univariate Analysis

Univariate analysis examines each feature individually to understand its distribution, frequency, central tendency, and variability. It helps identify common patterns, customer preferences, and operational characteristics within the dataset.

### 7.1.1 Numerical Feature Distribution

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns

df[num_cols].hist(
    figsize=(18,12),
    bins=30,
    edgecolor='black'
)

plt.suptitle(
    "Distribution of Numerical Features",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### 7.1.2 Distribution of Categorical Features

In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

print("Numerical Columns:")
print(list(num_cols))

print("\nCategorical Columns:")
print(list(cat_cols))
for col in cat_cols:

    if df[col].nunique()<=8:

        plt.figure(figsize=(6,6))

        df[col].value_counts().plot(
            kind="pie",
            autopct="%1.1f%%"
        )

        plt.ylabel("")

        plt.title(col)

        plt.show()

### 7.1.3 Boxplots of Numerical Features

In [ ]:
for col in num_cols:

    plt.figure(figsize=(10,2.8))

    sns.boxplot(
        x=df[col],
        color="#4C72B0"
    )

    plt.title(f"{col} Boxplot")

    plt.tight_layout()

    plt.show()

### 7.1.4 Correlation Preview

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    df[num_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")

plt.show()

### 7.1.5 Descriptive Statistics

In [ ]:
display(
    df.describe().T
)

### 7.1.6 Skewness

In [ ]:
skew = pd.DataFrame({
    "Skewness":df[num_cols].skew()
})

display(skew.sort_values("Skewness",ascending=False))

### 7.1.7 Kurtosis

In [ ]:
kurt = pd.DataFrame({
    "Kurtosis":df[num_cols].kurt()
})

display(kurt.sort_values("Kurtosis",ascending=False))

### 7.1.8 Value Counts of Categorical Features

In [ ]:
for col in cat_cols:

    print("="*60)
    print(col.upper())
    print("="*60)

    display(df[col].value_counts())

### 7.1.9 Pie Charts for Low-Cardinality Features

In [ ]:
for col in cat_cols:

    if df[col].nunique()<=8:

        plt.figure(figsize=(6,6))

        df[col].value_counts().plot(
            kind="pie",
            autopct="%1.1f%%"
        )

        plt.ylabel("")

        plt.title(col)

        plt.show()

## 7.2 Bivariate Analysis
### 7.2.1 Gender vs Rating

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="gender",
    y="rating",
    palette="Set2"
)

plt.title("Customer Rating by Gender")
plt.show()

### 7.2.2 Age vs Rating

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="age",
    y="rating",
    alpha=0.6
)

plt.title("Age vs Customer Rating")

plt.show()

### 7.2.3 Fare vs Distance

In [ ]:
df["fare_inr"] = (
    df["fare_inr"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("INR", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["fare_inr"] = pd.to_numeric(df["fare_inr"], errors="coerce")

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="distance_km",
    y="fare_inr",
    hue="payment_mode",
    alpha=0.7
)

plt.title("Fare vs Distance", fontsize=14, fontweight="bold")
plt.xlabel("Distance (km)")
plt.ylabel("Fare (INR)")
plt.legend(title="Payment Mode", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 7.2.4 City-wise Revenue

In [ ]:
city_rev = (
    df.groupby("city")["fare_inr"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(12,5))

city_rev.plot(kind="bar")

plt.ylabel("Revenue (INR)")
plt.title("Revenue by City")

plt.show()

### 7.2.5 Booking Channel Revenue


In [ ]:
channel = (
    df.groupby("booking_channel")["fare_inr"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
channel.plot(kind="bar")
plt.title("Revenue by Booking Channel")
plt.xlabel("Booking Channel")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 7.2.6 Payment Mode Revenue


In [ ]:
payment = (
    df.groupby("payment_mode")["fare_inr"]
      .sum()
      .sort_values(ascending=False)
)

payment.plot(
    kind="pie",
    autopct="%1.1f%%",
    figsize=(6, 6)
)

plt.ylabel("")
plt.title("Revenue Share by Payment Mode")
plt.tight_layout()
plt.show()


### 7.2.7 Route-wise Revenue


In [ ]:
route = (
    df.groupby("route_name")["fare_inr"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

plt.figure(figsize=(12, 5))

sns.barplot(
    x=route.values,
    y=route.index,
    palette="viridis"
)

plt.title("Top 10 Routes by Revenue")
plt.xlabel("Revenue (INR)")
plt.ylabel("Route Name")
plt.tight_layout()
plt.show()


### 7.2.8 Occupancy vs Rating


In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="occupancy_pct",
    y="rating",
    alpha=0.6
)

plt.title("Occupancy vs Customer Rating")
plt.xlabel("Occupancy (%)")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()


### 7.2.9 Weather vs Trip Status


In [ ]:
plt.figure(figsize=(9, 5))

sns.countplot(
    data=df,
    x="weather",
    hue="trip_status"
)

plt.title("Weather vs Trip Status")
plt.xlabel("Weather")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 7.2.10 Subscription Type vs Revenue


In [ ]:
sub = (
    df.groupby("subscription_type")["fare_inr"]
      .sum()
      .sort_values(ascending=False)
)

sub.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Revenue by Subscription Type")
plt.xlabel("Subscription Type")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7.3 Multivariate Analysis
### 7.3.1 Correlation Heatmap


In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    df.select_dtypes("number").corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


### 7.3.2 Pair Plot


In [ ]:
sns.pairplot(
    df[
        [
            "age",
            "distance_km",
            "fare_inr",
            "rating",
            "occupancy_pct"
        ]
    ]
)

plt.show()


### 7.3.3 Revenue by City and Payment Mode


In [ ]:
pivot = pd.pivot_table(
    df,
    values="fare_inr",
    index="city",
    columns="payment_mode",
    aggfunc="sum",
    fill_value=0
)

pivot.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Revenue by City and Payment Mode")
plt.xlabel("City")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 7.3.4 Average Rating by City


In [ ]:
rating = (
    df.groupby("city")["rating"]
      .mean()
      .sort_values(ascending=False)
)

rating.plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Average Rating by City")
plt.xlabel("City")
plt.ylabel("Average Rating")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 7.3.5 Occupancy by Bus Type


In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="bus_type",
    y="occupancy_pct"
)

plt.title("Occupancy Distribution by Bus Type")
plt.xlabel("Bus Type")
plt.ylabel("Occupancy (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 7.3.6 Revenue Heatmap


In [ ]:
heat = pd.pivot_table(
    df,
    values="fare_inr",
    index="city",
    columns="trip_status",
    aggfunc="sum",
    fill_value=0
)

plt.figure(figsize=(10, 6))

sns.heatmap(
    heat,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu"
)

plt.title("Revenue Heatmap by City and Trip Status")
plt.xlabel("Trip Status")
plt.ylabel("City")
plt.tight_layout()
plt.show()


## 8. Business Insights (KPIs)

This section converts the cleaned operational dataset into business-facing KPIs. The goal is to summarize revenue strength, demand concentration, service reliability, customer experience, and channel/payment behavior in a format that can support management decisions.


In [ ]:
numeric_columns = ["fare_inr", "rating", "distance_km", "occupancy_pct", "is_peak_hour"]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column].astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
            errors="coerce"
        )

total_trips = len(df)
total_revenue = df["fare_inr"].sum()
avg_fare = df["fare_inr"].mean()
avg_rating = df["rating"].mean()
avg_distance = df["distance_km"].mean()
avg_occupancy = df["occupancy_pct"].mean()
completion_rate = df["trip_status"].eq("Completed").mean() * 100
cancellation_rate = df["trip_status"].str.contains("cancel", case=False, na=False).mean() * 100
complaint_rate = df["complaint_raised"].eq("Yes").mean() * 100
peak_hour_share = df["is_peak_hour"].eq(1).mean() * 100
digital_payment_share = df["payment_mode"].isin(["UPI", "Upi", "Card", "Wallet", "Net Banking"]).mean() * 100

kpi_summary = pd.DataFrame(
    {
        "Metric": [
            "Total Trips",
            "Total Revenue (INR)",
            "Average Fare (INR)",
            "Average Rating",
            "Average Distance (km)",
            "Average Occupancy (%)",
            "Trip Completion Rate (%)",
            "Cancellation Rate (%)",
            "Complaint Rate (%)",
            "Peak-Hour Trip Share (%)",
            "Digital Payment Share (%)"
        ],
        "Value": [
            total_trips,
            total_revenue,
            avg_fare,
            avg_rating,
            avg_distance,
            avg_occupancy,
            completion_rate,
            cancellation_rate,
            complaint_rate,
            peak_hour_share,
            digital_payment_share
        ]
    }
)

display(kpi_summary.style.format({"Value": "{:,.2f}"}))

print("Top City by Bookings")
display(df["city"].value_counts().head(10).to_frame("bookings"))

print("Top Route by Bookings")
display(df["route_name"].value_counts().head(10).to_frame("bookings"))

print("Revenue by Payment Mode")
display(df.groupby("payment_mode")["fare_inr"].agg(trips="count", revenue="sum", avg_fare="mean").sort_values("revenue", ascending=False))

print("Trip Status Mix")
display(df["trip_status"].value_counts(normalize=True).mul(100).round(2).to_frame("share_pct"))


### 8.1 City and Route Performance Summary


In [ ]:
city_summary = (
    df.groupby("city")
      .agg(
          total_trips=("fare_inr", "count"),
          total_revenue=("fare_inr", "sum"),
          avg_fare=("fare_inr", "mean"),
          avg_rating=("rating", "mean"),
          avg_occupancy=("occupancy_pct", "mean"),
          completion_rate=("trip_status", lambda x: x.eq("Completed").mean() * 100),
          complaint_rate=("complaint_raised", lambda x: x.eq("Yes").mean() * 100)
      )
      .sort_values("total_revenue", ascending=False)
)

display(city_summary)

route_summary = (
    df.groupby("route_name")
      .agg(
          total_trips=("fare_inr", "count"),
          total_revenue=("fare_inr", "sum"),
          avg_rating=("rating", "mean"),
          avg_occupancy=("occupancy_pct", "mean"),
          completion_rate=("trip_status", lambda x: x.eq("Completed").mean() * 100),
          complaint_rate=("complaint_raised", lambda x: x.eq("Yes").mean() * 100)
      )
      .sort_values("total_revenue", ascending=False)
      .head(10)
)

display(route_summary)

bus_type_summary = (
    df.groupby("bus_type")
      .agg(
          total_trips=("fare_inr", "count"),
          total_revenue=("fare_inr", "sum"),
          avg_fare=("fare_inr", "mean"),
          avg_rating=("rating", "mean"),
          avg_occupancy=("occupancy_pct", "mean")
      )
      .sort_values("total_revenue", ascending=False)
)

display(bus_type_summary)


## 9. Executive Dashboard (Plotly)


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=(
        "Bookings by City",
        "Revenue by Payment Mode",
        "Trip Status Mix",
        "Customer Ratings",
        "Revenue by Bus Type",
        "Average Occupancy by City"
    ),
    specs=[
        [{"type": "bar"}, {"type": "pie"}],
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "bar"}, {"type": "bar"}]
    ],
    vertical_spacing=0.12
)

city = df["city"].value_counts().sort_values(ascending=False)

fig.add_trace(
    go.Bar(
        x=city.index,
        y=city.values,
        name="Bookings",
        marker_color="#2563eb"
    ),
    row=1,
    col=1
)

payment = df.groupby("payment_mode")["fare_inr"].sum().sort_values(ascending=False)

fig.add_trace(
    go.Pie(
        labels=payment.index,
        values=payment.values,
        name="Payment Mode",
        hole=0.35
    ),
    row=1,
    col=2
)

status = df["trip_status"].value_counts().sort_values(ascending=False)

fig.add_trace(
    go.Bar(
        x=status.index,
        y=status.values,
        name="Trip Status",
        marker_color="#16a34a"
    ),
    row=2,
    col=1
)

rating = df["rating"].value_counts().sort_index()

fig.add_trace(
    go.Bar(
        x=rating.index,
        y=rating.values,
        name="Ratings",
        marker_color="#f59e0b"
    ),
    row=2,
    col=2
)

bus_revenue = df.groupby("bus_type")["fare_inr"].sum().sort_values(ascending=False)

fig.add_trace(
    go.Bar(
        x=bus_revenue.index,
        y=bus_revenue.values,
        name="Bus Type Revenue",
        marker_color="#7c3aed"
    ),
    row=3,
    col=1
)

city_occupancy = df.groupby("city")["occupancy_pct"].mean().sort_values(ascending=False)

fig.add_trace(
    go.Bar(
        x=city_occupancy.index,
        y=city_occupancy.values,
        name="Average Occupancy",
        marker_color="#0891b2"
    ),
    row=3,
    col=2
)

fig.update_yaxes(title_text="Bookings", row=1, col=1)
fig.update_yaxes(title_text="Trips", row=2, col=1)
fig.update_yaxes(title_text="Customers", row=2, col=2)
fig.update_yaxes(title_text="Revenue (INR)", row=3, col=1)
fig.update_yaxes(title_text="Occupancy (%)", row=3, col=2)

fig.update_layout(
    height=1050,
    title_text="CityFlo Executive Dashboard",
    showlegend=False,
    template="plotly_white",
    margin=dict(t=90, l=50, r=40, b=60)
)

fig.show()


## 10. Key Findings


In [ ]:
top_city = df["city"].value_counts().idxmax()
top_city_share = df["city"].value_counts(normalize=True).max() * 100
top_route = df["route_name"].value_counts().idxmax()
top_route_share = df["route_name"].value_counts(normalize=True).max() * 100
top_payment_mode = df["payment_mode"].value_counts().idxmax()
top_booking_channel = df["booking_channel"].value_counts().idxmax()
highest_revenue_city = df.groupby("city")["fare_inr"].sum().idxmax()
highest_revenue_route = df.groupby("route_name")["fare_inr"].sum().idxmax()
highest_rating_city = df.groupby("city")["rating"].mean().idxmax()
lowest_rating_city = df.groupby("city")["rating"].mean().idxmin()
highest_occupancy_city = df.groupby("city")["occupancy_pct"].mean().idxmax()
lowest_occupancy_city = df.groupby("city")["occupancy_pct"].mean().idxmin()
best_bus_type = df.groupby("bus_type")["fare_inr"].sum().idxmax()
completion_rate = df["trip_status"].eq("Completed").mean() * 100
complaint_rate = df["complaint_raised"].eq("Yes").mean() * 100
avg_occupancy = df["occupancy_pct"].mean()
avg_rating = df["rating"].mean()

print("Key Findings")
print("-" * 80)
print(f"1. Demand is strongest in {top_city}, which contributes {top_city_share:.2f}% of total bookings.")
print(f"2. The most frequently used route is {top_route}, accounting for {top_route_share:.2f}% of all trips.")
print(f"3. {highest_revenue_city} generates the highest city-level revenue, while {highest_revenue_route} is the top revenue route.")
print(f"4. {top_payment_mode} is the most preferred payment mode, showing where payment convenience is currently strongest.")
print(f"5. {top_booking_channel} is the leading booking channel and should be treated as a priority customer touchpoint.")
print(f"6. The overall trip completion rate is {completion_rate:.2f}%, which is the main reliability KPI to track over time.")
print(f"7. Average occupancy is {avg_occupancy:.2f}%; {highest_occupancy_city} has the strongest occupancy and {lowest_occupancy_city} has the weakest.")
print(f"8. Average customer rating is {avg_rating:.2f}; {highest_rating_city} performs best on satisfaction and {lowest_rating_city} needs service attention.")
print(f"9. Complaint rate is {complaint_rate:.2f}%, so complaints should be monitored by route, driver, bus type, and weather condition.")
print(f"10. {best_bus_type} contributes the highest revenue among bus types and can guide fleet allocation decisions.")


## 11. Recommendations

1. Focus marketing and service expansion on high-revenue cities and routes.
2. Improve low-rated routes by reviewing delays, occupancy, and customer feedback.
3. Promote the strongest booking channel while improving weaker channels.
4. Track cancellation and failure patterns across weather and trip status.
5. Use occupancy trends to optimize bus allocation during peak and low-demand periods.
6. Encourage digital payment modes if they contribute strong revenue and convenience.


## 12. Conclusion

This exploratory data analysis converted the CityFlo Bus Service dataset into a structured view of demand, revenue, customer behavior, operational performance, and satisfaction. After cleaning and standardizing the data, the analysis showed how bookings and revenue vary across cities, routes, bus types, payment modes, booking channels, trip status, and customer ratings.

The business insights indicate that CityFlo should manage performance at both the city and route level. High-demand and high-revenue routes can be prioritized for frequency improvements, fleet allocation, subscription offers, and corporate partnerships. Routes or cities with weaker ratings, lower occupancy, higher complaints, or lower completion rates should be reviewed for schedule reliability, driver performance, bus quality, stop coverage, weather disruption, and customer support issues.

The executive dashboard provides a compact management view of the most important KPIs: bookings by city, revenue by payment mode, trip status mix, customer rating distribution, revenue by bus type, and occupancy by city. Together, these metrics help identify where CityFlo is performing well, where operational leakage may exist, and where growth investments are most likely to produce returns.

Overall, the cleaned dataset is suitable for further business intelligence, forecasting, route optimization, churn analysis, and predictive modeling. With regular monitoring of completion rate, occupancy, complaints, revenue per route, and customer satisfaction, CityFlo can make more data-driven decisions to improve service reliability, strengthen customer experience, and scale its metro city operations more efficiently.
